In [0]:
from pyspark.sql.functions import col

# Carregando as tabelas finais
df_vertices = spark.table("gold_vertices")
df_arestas = spark.table("gold_arestas")

print("Tabelas carregadas para validação.")

print("--- 1. TESTE DE VALORES NULOS ---")
nulos_vertices = df_vertices.filter(col("id_no").isNull()).count()
print(f"Vértices com ID nulo: {nulos_vertices} (Esperado: 0)")

nulos_arestas = df_arestas.filter(col("id_origem").isNull() | col("id_destino").isNull()).count()
print(f"Arestas com IDs nulos: {nulos_arestas} (Esperado: 0)")

if nulos_vertices == 0 and nulos_arestas == 0:
    print(" PASSED: Nenhum ID nulo encontrado.")
else:
    print(" FAILED: Valores nulos detectados!")

    print("\n--- 2. TESTE DE DUPLICATAS ---")
total_vertices = df_vertices.count()
vertices_distintos = df_vertices.select("id_no").distinct().count()
duplicatas = total_vertices - vertices_distintos

print(f"Vértices duplicados: {duplicatas} (Esperado: 0)")
if duplicatas == 0:
    print(" PASSED: Chave primária é única.")
else:
    print(" FAILED: Duplicatas detectadas!")

    print("\n--- 3. TESTE DE INTEGRIDADE REFERENCIAL ---")
    
# Todos os IDs de Origem (ex: Autores) estão cadastrados nos Vértices?
arestas_origem = df_arestas.select("id_origem").distinct()
vertices_id = df_vertices.select("id_no").distinct()

orfãos_origem = arestas_origem.join(vertices_id, arestas_origem.id_origem == vertices_id.id_no, "left_anti").count()
print(f"IDs de Origem não encontrados nos Vértices: {orfãos_origem} (Esperado: 0)")

# Todos os IDs de Destino (ex: Trabalhos Citados) estão cadastrados nos Vértices?
arestas_destino = df_arestas.select("id_destino").distinct()
orfãos_destino = arestas_destino.join(vertices_id, arestas_destino.id_destino == vertices_id.id_no, "left_anti").count()

print(f"IDs de Destino não encontrados nos Vértices: {orfãos_destino}")
print(" NOTA: Se o valor acima for MAIOR que zero, não é um erro de engenharia! Isso ocorre porque estamos usando uma AMOSTRA de 1.000 artigos. Se o Artigo A cita o Artigo B, mas o Artigo B não foi baixado na amostra, ele não estará na tabela de Vértices.")